In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [12]:
gl= pd.read_csv("general_ledger_2024_2025.csv",sep=";")
coa       = pd.read_csv("chart_of_accounts.csv")
cc        = pd.read_csv("cost_centers.csv")
customers = pd.read_csv("customers.csv")
budget    = pd.read_excel("budget_2024_2025.xlsx")
invoices  = pd.read_excel("accounts_receivable_invoices.xlsx")


In [13]:
gl.head(5)

,journal_id,posting_date,account_no,cost_center,amount,currency,vendor,description
0,JE101094,2025-01-03,4000,CC100,11406.71,EUR,NaN,Revenue - Machine Sales Jan 2025
1,JE102157,11.12.2025,6400,CC300,-19410.54,EUR,Google Ireland Ltd,Marketing & Advertising Dec 2025
2,JE100821,2024-09-07,6700,CC600,-1267.12,EUR,KPMG AG,Consulting & Legal Sep 2024
3,JE101506,2025-05-08,6100,CC200,-14210.48,EUR,NaN,Social Security Contributions May 2025
4,JE100443,23.05.2024,6300,CC200,-1368.13,EUR,Vattenfall Europe,Utilities May 2024


In [14]:
gl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   journal_id    2225 non-null   object
 1   posting_date  2225 non-null   object
 2   account_no    2225 non-null   int64 
 3   cost_center   2170 non-null   object
 4   amount        2225 non-null   object
 5   currency      2225 non-null   object
 6   vendor        1477 non-null   object
 7   description   2225 non-null   object
dtypes: int64(1), object(7)
memory usage: 139.2+ KB


In [19]:
gl[gl['amount'].str.contains(',',na=False)].head().sum()

journal_id               JE100364JE101265JE101275JE101249JE100538
posting_date    2024-04-072025-02-252025-02-232025-02-012024-0...
account_no                                                  32600
cost_center                             CC400CC600CC500CC500CC110
amount                   -19635,17-1204,73-3156,62-123,35-1056,07
currency                                          EUREUREUREUREUR
vendor          Microsoft DeutschlandLufthansaKPMG AGVattenfal...
description     IT & Software Apr 2024Travel Expenses Feb 2025...
dtype: object

In [ ]:
###converting values to float

In [21]:
gl['amount'].str.contains(',',na=False).sum()

np.int64(60)

In [23]:
gl['amount']=gl['amount'].astype(str).str.replace(',','.').astype(float)

In [24]:
print(gl['amount'].dtype)

float64


In [25]:
gl['posting_date'].sample(15, random_state=1)

1664    2025-05-16
1347    2024-02-08
432     2024-04-14
1435    05.11.2024
1374    2024-12-16
1610    2024-01-10
1330    2024-01-03
1220    2025-02-20
661     2025-01-17
553     07/09/2025
1152    2025-12-24
495     2025-07-08
1891    2024-09-09
615     2024-04-08
1707    2025-10-27
Name: posting_date, dtype: object

In [26]:
d1 = pd.to_datetime(gl['posting_date'], format='%Y-%m-%d', errors='coerce')   # 2025-01-03
d2 = pd.to_datetime(gl['posting_date'], format='%d.%m.%Y', errors='coerce')   # 03.01.2025
d3 = pd.to_datetime(gl['posting_date'], format='%d/%m/%Y', errors='coerce')   # 03/01/2025

gl['posting_date'] = d1.fillna(d2).fillna(d3)

print("Missing dates:", gl['posting_date'].isna().sum())
print(gl['posting_date'].dtype)

Missing dates: 0
datetime64[ns]


In [29]:
gl['posting_date'].sample(10)

968    2025-01-21
1998   2024-10-28
1536   2025-06-04
1794   2024-02-05
2135   2025-08-19
2168   2024-09-02
1848   2024-06-09
1771   2024-04-19
118    2024-05-01
154    2024-01-28
Name: posting_date, dtype: datetime64[ns]

In [30]:
gl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   journal_id    2225 non-null   object        
 1   posting_date  2225 non-null   datetime64[ns]
 2   account_no    2225 non-null   int64         
 3   cost_center   2170 non-null   object        
 4   amount        2225 non-null   float64       
 5   currency      2225 non-null   object        
 6   vendor        1477 non-null   object        
 7   description   2225 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 139.2+ KB


In [31]:
gl.isna().sum()

journal_id        0
posting_date      0
account_no        0
cost_center      55
amount            0
currency          0
vendor          748
description       0
dtype: int64

In [34]:
gl.duplicated().sum()

np.int64(33)

In [35]:
print(gl['currency'].value_counts(), "\n")
print(gl['cost_center'].value_counts(dropna=False), "\n")
print(gl['vendor'].value_counts())

currency
EUR    2199
USD      26
Name: count, dtype: int64 

cost_center
CC500    436
CC200    423
CC110    311
CC100    288
CC600    214
CC210    208
CC400    135
CC300    125
NaN       55
cc200      9
cc500      7
cc100      7
cc210      2
cc600      2
cc110      1
cc400      1
cc300      1
Name: count, dtype: int64 

vendor
Hays AG                     193
Vattenfall Europe           176
KPMG AG                     176
Lufthansa                   139
Deutsche Bahn               136
DHL Freight                  88
Siemens Komponenten          83
Staples Office               83
Commerzbank AG               82
Google Ireland Ltd           80
Bosch Rexroth AG             56
Microsoft Deutschland        46
Thyssen Stahl GmbH           45
SAP Deutschland              42
Immobilien Berlin GmbH       25
                              6
Vattenfall Europe             3
DHL Freight                   2
Google Ireland Ltd            2
Hays AG                       2
Siemens Komponenten           2

In [39]:
gl['cost_center'] = gl['cost_center'].str.upper().str.strip()
gl['vendor'] = gl['vendor'].str.strip()
print(gl['cost_center'].value_counts(dropna=False))
print("\nDuplicate rows now:", gl.duplicated().sum())

cost_center
CC500    443
CC200    432
CC110    312
CC100    295
CC600    216
CC210    210
CC400    136
CC300    126
NaN       55
Name: count, dtype: int64

Duplicate rows now: 36


In [42]:
usd = gl['currency'] == 'USD'
print("USD rows:", usd.sum())
gl.loc[usd, 'amount'] = (gl.loc[usd, 'amount'] / 1.09).round(2)
gl.loc[usd, 'currency'] = 'EUR'

gl['currency'].value_counts()

USD rows: 0


currency
EUR    2225
Name: count, dtype: int64

In [47]:
print("Duplicate journas id", gl['journal_id'].duplicated().sum())

Duplicate journas id 40


In [52]:
rows_before=len(gl)
print (rows_before)

2185


In [53]:
gl = gl.sort_values('cost_center', na_position='last')
gl = gl.drop_duplicates(subset='journal_id', keep='first')
gl = gl.sort_index()
print("Removed:", rows_before - len(gl), "| Rows now:", len(gl))

Removed: 0 | Rows now: 2185


In [55]:
print("Duplicate journal IDs now:", gl['journal_id'].duplicated().sum())



Duplicate journal IDs now: 0


In [59]:
test_rows = gl['description'].str.contains('Test', case=False)
rows_before = len(gl)
gl = gl[~test_rows]
print("Removed:", rows_before - len(gl), "| Rows now:", len(gl))


Removed: 2 | Rows now: 2183


In [61]:
gl['cost_center'].isna().sum()

np.int64(52)

In [63]:
gl['cost_center']=gl['cost_center'].fillna('UNASSIGNED')

In [64]:
gl['cost_center'].value_counts()

cost_center
CC500         435
CC200         424
CC110         308
CC100         287
CC600         214
CC210         205
CC400         133
CC300         125
UNASSIGNED     52
Name: count, dtype: int64

In [65]:
unknown = ~gl['account_no'].isin(coa['account_no'])

In [67]:
gl[unknown]

,journal_id,posting_date,account_no,cost_center,amount,currency,vendor,description
374,JE100834,2024-09-07,6450,CC500,-673.03,EUR,Commerzbank AG,Interest Expense Sep 2024
639,JE101502,2025-05-20,6450,CC500,-23083.53,EUR,Hays AG,Salaries May 2025
687,JE101489,2025-05-20,6450,CC110,-1516.43,EUR,NaN,Sales Discounts May 2025
1192,JE101190,2025-02-24,6450,CC100,91497.95,EUR,NaN,Revenue - Machine Sales Feb 2025
1340,JE100473,2024-05-20,6450,CC500,-1666.98,EUR,Commerzbank AG,Interest Expense May 2024
1442,JE102153,2025-12-01,6450,CC200,-171.48,EUR,Vattenfall Europe,Utilities Dec 2025
1550,JE100601,2024-07-18,6450,CC300,-17083.53,EUR,Hays AG,Salaries Jul 2024
2121,JE100696,2024-08-25,6450,CC100,-6493.99,EUR,NaN,Social Security Contributions Aug 2024


In [69]:
gl['flag_unknown_account'] = unknown

In [70]:
gl.to_csv("gl_clean.csv", index=False)

In [71]:
budget.head()

,fiscal_year,month,cost_center,account_no,budget_eur
0,2024,1,CC100,4000,297500
1,2024,1,CC110,4000,297500
2,2024,1,CC210,4100,119000
3,2024,1,CC210,4200,80800
4,2024,1,CC100,4900,-7600


In [72]:
budget.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 936 entries, 0 to 935
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   fiscal_year  936 non-null    int64 
 1   month        936 non-null    int64 
 2   cost_center  936 non-null    object
 3   account_no   936 non-null    int64 
 4   budget_eur   936 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 36.7+ KB


In [73]:
budget.duplicated().sum()

np.int64(0)

In [74]:
budget['cost_center'].isin(cc['cost_center']).all()

np.True_

In [75]:
budget['account_no'].isin(coa['account_no']).all()


np.True_

In [76]:
budget.groupby('fiscal_year')['month'].unique()

fiscal_year
2024    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
2025    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Name: month, dtype: object

In [77]:
budget.to_csv("budget_clean.csv", index=False)

In [78]:
invoices.head()

,invoice_no,customer_id,invoice_date,due_date,payment_date,amount_eur,status
0,INV-24001,C1003,2024-12-08,2025-01-07,2025-01-26,20711.18,Paid
1,INV-24002,C1010,2024-01-11,2024-02-25,2024-02-07,34797.66,Paid
2,INV-24003,C1017,2025-02-11,2025-03-13,2025-04-08,13656.86,Paid
3,INV-24004,C1016,2025-10-02,2025-12-01,2025-12-02,28265.78,Paid
4,INV-24005,C1017,2025-09-16,2025-10-16,2025-11-05,22288.53,Paid


In [79]:
invoices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   invoice_no    900 non-null    object        
 1   customer_id   900 non-null    object        
 2   invoice_date  900 non-null    datetime64[ns]
 3   due_date      900 non-null    datetime64[ns]
 4   payment_date  853 non-null    datetime64[ns]
 5   amount_eur    900 non-null    float64       
 6   status        900 non-null    object        
dtypes: datetime64[ns](3), float64(1), object(3)
memory usage: 49.3+ KB


In [80]:
invoices['status'].value_counts()

status
Paid        853
Open         28
Disputed     19
Name: count, dtype: int64

In [81]:
invoices[invoices['payment_date'].isna()]['status'].value_counts()

status
Open        28
Disputed    19
Name: count, dtype: int64

In [85]:
invoices['invoice_no'].duplicated().sum()

np.int64(0)

In [87]:
invoices['customer_id'].isin(customers['customer_id']).all()

np.True_

In [88]:
invoices['amount_eur'].describe()

count       900.000000
mean      19529.347778
std       15707.250693
min        1337.580000
25%        9012.422500
50%       15083.720000
75%       25382.777500
max      130909.900000
Name: amount_eur, dtype: float64

In [92]:
(invoices['due_date']<invoices['invoice_date']).sum()

np.int64(0)

In [96]:
invoices['days_late']=(invoices['payment_date']-invoices['due_date']).dt.days

In [97]:
invoices.head()

,invoice_no,customer_id,invoice_date,due_date,payment_date,amount_eur,status,days_late
0,INV-24001,C1003,2024-12-08,2025-01-07,2025-01-26,20711.18,Paid,19.0
1,INV-24002,C1010,2024-01-11,2024-02-25,2024-02-07,34797.66,Paid,-18.0
2,INV-24003,C1017,2025-02-11,2025-03-13,2025-04-08,13656.86,Paid,26.0
3,INV-24004,C1016,2025-10-02,2025-12-01,2025-12-02,28265.78,Paid,1.0
4,INV-24005,C1017,2025-09-16,2025-10-16,2025-11-05,22288.53,Paid,20.0


In [98]:
invoices.to_csv("invoices_clean.csv", index=False)

In [99]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   customer_id         25 non-null     object
 1   customer_name       25 non-null     object
 2   country             25 non-null     object
 3   segment             25 non-null     object
 4   payment_terms_days  25 non-null     int64 
 5   credit_limit_eur    25 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 1.3+ KB


In [100]:
customers['customer_id'].duplicated().sum()

np.int64(0)

In [101]:
customers.head()

,customer_id,customer_name,country,segment,payment_terms_days,credit_limit_eur
0,C1001,Volkswagen Zulieferung,DE,Automotive,60,500000
1,C1002,BASF Logistik,DE,Automotive,30,500000
2,C1003,Krones AG,DE,Agriculture,30,100000
3,C1004,Claas KGaA,DE,Automotive,45,250000
4,C1005,Liebherr Werk Ehingen,DE,Industrial,60,100000


In [102]:
customers['payment_terms_days'].value_counts()

payment_terms_days
30    13
60     8
45     4
Name: count, dtype: int64

In [103]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   customer_id         25 non-null     object
 1   customer_name       25 non-null     object
 2   country             25 non-null     object
 3   segment             25 non-null     object
 4   payment_terms_days  25 non-null     int64 
 5   credit_limit_eur    25 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 1.3+ KB


In [104]:
customers['customer_id'].duplicated().sum()

np.int64(0)

In [105]:
customers.to_csv("customers_clean.csv", index=False)

In [106]:
coa.to_csv("coa_clean.csv", index=False)
cc.to_csv("cost_centers_clean.csv", index=False)